In [5]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_distances

# load data

In [6]:
df = pd.read_csv('/Users/connorhall/datasets/inst414/module 3 assignment/Public.csv', usecols=['SPECIES', 'AIRPORT', 'STATE'])
df = df.dropna()

In [7]:
# remove unknown species
df = df[(~df['SPECIES'].str.contains('unknown|perching birds', case=False)) & (~df['AIRPORT'].str.contains('unknown', case=False))]
# species must have >= 100 collisions
df = df[df.groupby('SPECIES')['SPECIES'].transform('count') >= 100]
df

,AIRPORT,STATE,SPECIES
8,NORMAN Y. MINETA SAN JOSE INTL ARPT,CA,American robin
11,ALEXANDRIA INTL,LA,Blackbirds
13,SYRACUSE HANCOCK INTL,NY,Gulls
14,OAKLAND COUNTY INTL,MI,Mourning dove
15,LOS ANGELES INTL,CA,American kestrel
...,...,...,...
339266,EVANSVILLE REGIONAL,IN,Chimney swift
339267,SEATTLE-TACOMA INTL,WA,Cedar waxwing
339268,CHICAGO O'HARE INTL ARPT,IL,Yellow-rumped warbler
339269,ROANOKE REGNL ARPT/WOODRUM FIELD,VA,European starling


In [8]:
len(df['SPECIES'].unique())

174

# calculate collision counts

In [9]:
collision_count = pd.crosstab(df['SPECIES'], df['AIRPORT']).stack('AIRPORT')
collision_count

SPECIES            AIRPORT                             
American barn owl  ABBEVILLE CHRIS CRUSTA MEMORIAL APRT    0
                   ABERDEEN REGIONAL ARPT                  0
                   ABERNATHY FIELD ARPT                    0
                   ABILENE REGIONAL ARPT                   0
                   ABRAHAM LINCOLN CAPITAL ARPT            0
                                                          ..
Zebra dove         ZAMPERINI FIELD ARPT                    0
                   ZANESVILLE MUNICIPAL ARPT               0
                   ZELIENOPLE MUNICIPAL                    0
                   ZEPHYRHILLS MUNICIPAL ARPT              0
                   ZURICH                                  0
Length: 371664, dtype: int64

### separate species and airport index

In [10]:
species = collision_count.index.get_level_values('SPECIES').tolist()
airports = collision_count.index.get_level_values('AIRPORT').tolist()

collisions_df = pd.DataFrame({'Species': species, 'Airport': airports})
collisions_df['Count'] = collision_count.values
collisions_df

,Species,Airport,Count
0,American barn owl,ABBEVILLE CHRIS CRUSTA MEMORIAL APRT,0
1,American barn owl,ABERDEEN REGIONAL ARPT,0
2,American barn owl,ABERNATHY FIELD ARPT,0
3,American barn owl,ABILENE REGIONAL ARPT,0
4,American barn owl,ABRAHAM LINCOLN CAPITAL ARPT,0
...,...,...,...
371659,Zebra dove,ZAMPERINI FIELD ARPT,0
371660,Zebra dove,ZANESVILLE MUNICIPAL ARPT,0
371661,Zebra dove,ZELIENOPLE MUNICIPAL,0
371662,Zebra dove,ZEPHYRHILLS MUNICIPAL ARPT,0


### add collision counts to new dataframe format

In [11]:
# create lists of unique species names and airport names
unique_sp = sorted(list(set(species)))
unique_air = sorted(list(set(airports)))

In [12]:
# create list of collision count lists
counts = []
for sp in unique_sp:
    sp_filter = collisions_df[collisions_df['Species'] == sp]

    counts.append(sp_filter['Count'].tolist())

In [13]:
# create new dataframe format
collisions_df = pd.DataFrame(counts, columns=unique_air, index=unique_sp)
pd.options.display.max_columns = None
collisions_df

ABBEVILLE CHRIS CRUSTA MEMORIAL APRT  \
American barn owl                                              0   
American coot                                                  0   
American crow                                                  0   
American golden-plover                                         0   
American goldfinch                                             0   
...                                                          ...   
Yellow warbler                                                 0   
Yellow-bellied sapsucker                                       0   
Yellow-crowned night heron                                     0   
Yellow-rumped warbler                                          0   
Zebra dove                                                     0   

                            ABERDEEN REGIONAL ARPT  ABERNATHY FIELD ARPT  \
American barn owl                                0                     0   
American coot                                    1                     0   
American crow                                    0                     0   
American golden-plover                           0                     0   
American goldfinch                               0                     0   
...                                            ...                   ...   
Yellow warbler                                   0                     0   
Yellow-bellied sapsucker                         0                     0   
Yellow-crowned night heron                       0                     0   
Yellow-rumped warbler                            0                     0   
Zebra dove                                       0                     0   

                            ABILENE REGIONAL ARPT  \
American barn owl                               0   
American coot                                   0   
American crow                                   0   
American golden-plover                          0   
American goldfinch                              0   
...                                           ...   
Yellow warbler                                  0   
Yellow-bellied sapsucker                        0   
Yellow-crowned night heron                      0   
Yellow-rumped warbler                           0   
Zebra dove                                      0   

                            ABRAHAM LINCOLN CAPITAL ARPT  \
American barn owl                                      0   
American coot                                          1   
American crow                                          1   
American golden-plover                                 0   
American goldfinch                                     1   
...                                                  ...   
Yellow warbler                                         0   
Yellow-bellied sapsucker                               0   
Yellow-crowned night heron                             0   
Yellow-rumped warbler                                  0   
Zebra dove                                             0   

                            ABRAMS MUNICIPAL ARPT  ACADIANA REGIONAL ARPT  \
American barn owl                               0                       0   
American coot                                   0                       0   
American crow                                   0                       0   
American golden-plover                          0                       0   
American goldfinch                              0                       0   
...                                           ...                     ...   
Yellow warbler                                  0                       0   
Yellow-bellied sapsucker                        0                       0   
Yellow-crowned night heron                      0                       0   
Yellow-rumped warbler                           0                       0   
Zebra dove                                      0                       0   

               

# analysis

### find species of interest- high collision counts

In [14]:
df['SPECIES'].value_counts().head(20)

SPECIES
Mourning dove         16854
Killdeer              10963
Barn swallow          10946
American kestrel      10132
Horned lark            8969
Gulls                  7391
European starling      6809
Eastern meadowlark     5027
Rock pigeon            4594
Red-tailed hawk        4366
Sparrows               4314
Cliff swallow          3357
Western meadowlark     2816
Ring-billed gull       2419
American barn owl      2288
Canada goose           2192
Herring gull           2189
American robin         1906
Microbats              1709
Hawks                  1656
Name: count, dtype: int64

In [15]:
target_species = ['American kestrel', 'Horned lark', 'Herring gull']

### top 10 similar species by cosine distance

In [16]:
dist = cosine_distances
dataframes = []

for species in target_species:

    # get collision counts for target species
    target_species_collisions = collisions_df.loc[species]

    #Generating distances from that species to all the others
    distances = dist(collisions_df, [target_species_collisions])[:,0]

    query_distances = list(zip(collisions_df.index, distances))

    # top 10 most similar species
    species_list = []
    score_list = []
    for similar_species, similar_airport_score in sorted(query_distances, key=lambda x: x[1], reverse=False)[:10]:
        #print(similar_species, similar_airport_score)
        species_list.append(similar_species)
        score_list.append(similar_airport_score)
    dataframes.append(pd.DataFrame({'Species': species_list, 'Similarity score': score_list}))
    

In [17]:
# american kestrel
dataframes[0]

,Species,Similarity score
0,American kestrel,0.000000
1,Barn swallow,0.330259
2,Ring-billed gull,0.349587
3,Mallard,0.352982
4,European starling,0.356551
5,Red-tailed hawk,0.360380
6,Peregrine falcon,0.383626
7,American crow,0.383828
8,Song sparrow,0.406108
9,American robin,0.406257


In [18]:
# horned lark
dataframes[1]

,Species,Similarity score
0,Horned lark,7.216450e-15
1,Gopher snake,3.197962e-02
2,Desert cottontail,3.262536e-02
3,White-tailed jackrabbit,3.412343e-02
4,Lark bunting,3.957095e-02
5,Black-tailed prairie dog,4.752286e-02
6,Western meadowlark,8.053998e-02
7,Great horned owl,8.341577e-02
8,Eastern cottontail,1.537068e-01
9,Cliff swallow,2.584500e-01


In [19]:
# american herring gull
dataframes[2]

,Species,Similarity score
0,Herring gull,0.000000
1,Great black-backed gull,0.034922
2,Double-crested cormorant,0.112587
3,Laughing gull,0.156362
4,Peregrine falcon,0.167948
5,Black-crowned night heron,0.196500
6,Northern flicker,0.214599
7,Osprey,0.223654
8,Wood duck,0.231591
9,Snow bunting,0.235810


# limitations

### strikes per state

In [20]:
df['STATE'].value_counts()[:10]

STATE
TX    16297
FL    14013
CA    12755
NY    10707
CO    10536
IL     8618
MI     6320
OH     6275
NJ     5704
PA     5299
Name: count, dtype: int64